# ZeRO

ZeRO（全称 Zero Redundancy Optimizer，零冗余优化器）是由 DeepSpeed 团队提出的一项革命性大模型分布式训练内存优化技术。
它的核心目标非常明确：消除数据并行训练中的显存冗余，用有限的 GPU 显存去训练无限大的模型。

## 🤔 为什么需要 ZeRO？（显存都去哪儿了）
在传统的分布式数据并行（DDP）训练中，每个 GPU 都会完整地保存一份模型的所有状态。随着模型参数量达到百亿、千亿级别，显存会被以下三个“大头”迅速吃光：
1. <font color='red'>模型参数 (Parameters)</font>：FP16 精度下，每个参数占 2 字节。
2. <font color='red'>梯度 (Gradients)</font>：反向传播计算得出，FP16 精度下同样占 2 字节/参数。
3. <font color='red'>优化器状态 (Optimizer States)</font>：<font color='green'>这是最大的显存杀手。</font>
    - <b>以常用的 Adam 优化器为例，它需要为每个参数维护 FP32 的主权重副本、动量（一阶矩）和方差（二阶矩），总共需要 12 字节/参数。</b>

举个例子： <font color='red'>训练一个 70 亿（7B）参数的模型，光是优化器状态就需要约 84GB 显存</font>，加上参数和梯度，总需求远超单张 A100/H100 的 80GB 上限。ZeRO 就是为了解决这个问题而生的。

## ⚙️ ZeRO 的核心原理：分片（Sharding）
ZeRO 的核心思路非常简单粗暴：既然每个 GPU 都存一份完整的模型状态太浪费，那就把这些状态切碎（分片），分摊到所有参与训练的 GPU 上。 当某个 GPU 需要用到不属于自己的数据时，再通过高效的通信（如 All-Gather, Reduce-Scatter）临时获取。

根据分片对象的不同，ZeRO 分为三个循序渐进的阶段（Stage）：

1️⃣ ZeRO-1：优化器状态分片
- 做了什么：仅将占显存最大的优化器状态进行分片，<font color='red'>模型参数和梯度依然在每个 GPU 上保留完整副本。</font>
- 效果与代价：显存占用明显下降（大约节省 4 倍），通信开销与传统的数据并行（DDP）几乎完全相同。
- 适用场景：适合模型参数较大，且主要瓶颈在于优化器状态的场景（例如使用 Adam 进行全量微调或预训练）。

2️⃣ ZeRO-2：优化器状态 + 梯度分片
- 做了什么：在 ZeRO-1 的基础上，进一步将梯度也进行分片。
- 效果与代价：显存进一步压缩（大约节省 8 倍）。通信开销依然保持在与 DDP 同量级的水平，只是通信的方式从单纯的 All-Reduce 变成了 Reduce-Scatter 和 All-Gather 的组合。
- 适用场景：适合梯度占用显存较多，且希望在多节点训练中进一步压榨显存的场景。

3️⃣ ZeRO-3：全模型状态分片（参数+梯度+优化器）
- 做了什么：这是最极致的优化。它将模型参数、梯度、优化器状态全部切碎并分摊到各个 GPU 上。每个 GPU 平时只持有自己负责的那一小部分模型片段。
- 效果与代价：显存占用降至最低，理论上可以支撑起万亿参数级别的模型训练。但代价是通信开销大幅增加，因为在前向传播和反向传播的每一层，GPU 都需要通过通信去“借”其他 GPU 上的参数分片，计算完后再“还”回去。
- 适用场景：超大模型（千亿、万亿参数）的预训练，或者显存极度受限的多节点训练环境。

| 阶段 | 分片对象 | 显存优化效果 | 通信开销 |
| :--- | :--- | :--- | :--- |
| ZeRO-1 | 仅优化器状态 | ⭐⭐⭐ (较好) | 低 (与DDP持平) |
| ZeRO-2 | 优化器状态 + 梯度 | ⭐⭐⭐⭐ (优秀) | 低 (与DDP持平) |
| ZeRO-3 | 参数 + 梯度 + 优化器 | ⭐⭐⭐⭐⭐ (极致) | 高 (需频繁通信参数) |


## 💡 补充：ZeRO-R 与 ZeRO-Infinity
除了上述核心的 ZeRO-DP（数据并行优化），ZeRO 家族还有针对“残余状态”（Residual States，如激活值 Activation、临时缓冲区等）的优化技术，称为 ZeRO-R。它通常配合激活重计算（Activation Checkpointing）使用，进一步降低显存占用。

总结来说，ZeRO 技术通过“化整为零”的策略，打破了单卡显存对大模型训练的限制，让普通的研究者和工程师在没有极其昂贵的硬件条件下，也能高效地训练和微调超大参数量的语言模型。